<a href="https://colab.research.google.com/github/ntu-dl-bootcamp/deep-learning-2026/blob/main/SESSION2/solutions/session2_part3.ipynb" target="_blank"><img alt="Open In Colab" src="https://colab.research.google.com/assets/colab-badge.svg"/></a>

# Challenge - House Price Prediction Using Neural Network

Using the concepts learnt in the session so far, try to solve a classic regression problem - predicting the price of a house based on its features!

You can take the help of parts 1 and 2 of the notebook wherever necessary.

### Imports

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from torch import nn

### Set device

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


## Challenge 1: Dataset and DataLoader

### Loading Data

In [3]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing_data = fetch_california_housing()
# print(housing_data)
print("Number of Features: ", len(housing_data.data[0]))
print("Number of Data Samples: ", len(housing_data.data))

Number of Features:  8
Number of Data Samples:  20640


### Complete Custom Dataset Class

In [4]:
# Refer: https://pytorch.org/tutorials/beginner/data_loading_tutorial.html

class CustomDataset(Dataset):
    def __init__(self, data_source):
        self.data_source =data_source
        self.features = torch.tensor(self.data_source.data, dtype=torch.float32)
        self.labels = torch.tensor(self.data_source.target, dtype=torch.float32)


    def __len__(self):
        return len(self.data_source.data)

    def __getitem__(self, idx):
        feature, label = self.features[idx], self.labels[idx]  # Use idx (index) to reference each input row in the dataset
        return feature, label


### Specify the batch_size

In [5]:
batch_size = 50 # try different values like 1, 2, 10, 100

### Declare Custom Dataset Class Object

In [6]:
custom_dataset = CustomDataset(housing_data)

### Split the dataset into train and test

In [7]:
from torch.utils.data import random_split

split_ratio = 0.9 # 9:1 ratio for train:test
train_size = int(split_ratio * len(custom_dataset))
test_size = len(custom_dataset) - train_size

#Refer to torch.utils.data.random_split() in https://pytorch.org/docs/stable/data.html

housing_train_dataset, housing_test_dataset = random_split(custom_dataset, [train_size, test_size])

print(len(housing_train_dataset))
print(len(housing_test_dataset))


18576
2064


### Intialize the data loader

In [8]:
train_housing_data_loader = DataLoader(housing_train_dataset, batch_size=batch_size, shuffle=True)
test_housing_data_loader = DataLoader(housing_test_dataset, batch_size=batch_size, shuffle=True)

### Execute the below code to check Dataloader  - Try changing the batch_size!

In [9]:
for  id, batch_data in enumerate(train_housing_data_loader):

    batch_features, batch_labels = batch_data[0], batch_data[1]
    print("batch_features: ",  batch_features)
    print("batch_labels: ", batch_labels)

    break

batch_features:  tensor([[ 5.4273e+00,  4.4000e+01,  5.7120e+00,  9.5467e-01,  8.4600e+02,
          2.2560e+00,  3.7330e+01, -1.2193e+02],
        [ 3.0000e+00,  2.1000e+01,  4.5875e+00,  1.0832e+00,  1.5110e+03,
          1.8227e+00,  3.3770e+01, -1.1803e+02],
        [ 3.0833e+00,  2.5000e+01,  4.2729e+00,  1.0628e+00,  1.3710e+03,
          3.3116e+00,  3.3890e+01, -1.1835e+02],
        [ 4.5987e+00,  3.2000e+01,  5.2959e+00,  1.1779e+00,  1.3130e+03,
          2.4588e+00,  3.3920e+01, -1.1840e+02],
        [ 5.7500e+00,  1.6000e+01,  6.3000e+00,  1.1700e+00,  3.4300e+02,
          3.4300e+00,  3.4420e+01, -1.1923e+02],
        [ 6.3624e+00,  3.0000e+01,  5.6154e+00,  7.3077e-01,  1.2600e+02,
          2.4231e+00,  3.7810e+01, -1.2218e+02],
        [ 3.5288e+00,  8.0000e+00,  5.7335e+00,  1.1397e+00,  1.1590e+03,
          2.1305e+00,  3.4880e+01, -1.2041e+02],
        [ 2.9386e+00,  1.8000e+01,  4.8952e+00,  1.1380e+00,  4.0280e+03,
          3.5179e+00,  3.4060e+01, -1.1753e+02],

## Challenge 2: Building Neural Network

### Complete Code for NeuralNetwork class below

In [10]:
class HousingNeuralNetwork(nn.Module): # Inherit from nn.Module
    def __init__(self):
        super().__init__()


        # Define a sequential container of layers in network
        #You are free to choose number of layers and number of neurons in each layer
        self.layer_stack = nn.Sequential(
            nn.Linear(8, 4),
            nn.ReLU(),
            nn.Linear(4, 2),
            nn.ReLU(),
            nn.Linear(2, 1),
        )

    def forward(self, x):

        output = self.layer_stack(x)
        return output

#Create Object of HousingNeuralNetwork class
housing_model = HousingNeuralNetwork()

#Send model to DEVICE
housing_model = housing_model.to(DEVICE)

#print model
print(housing_model)

HousingNeuralNetwork(
  (layer_stack): Sequential(
    (0): Linear(in_features=8, out_features=4, bias=True)
    (1): ReLU()
    (2): Linear(in_features=4, out_features=2, bias=True)
    (3): ReLU()
    (4): Linear(in_features=2, out_features=1, bias=True)
  )
)


## Challenge 3 - Network Training

### Specify Hyperparameter - learning rate and number of epochs

In [11]:
learning_rate = 1e-3 # Try different values like 0.01 , 0.002. etc
epochs = 5

### Specify the loss function

In [12]:
housing_loss_fn = nn.MSELoss()

### Specify Optimizer

In [13]:
from torch import optim

housing_optimizer =  optim.SGD(housing_model.parameters(), lr=learning_rate)

### Complete code for Train loop

In [14]:
def housing_train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    loss = 0
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        X=X.to(DEVICE)
        y=y.to(DEVICE)
        pred = model(X).flatten()
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)

            print(f"Train loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

    return loss

### Complete code for Test loop

In [15]:
def housing_test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X=X.to(DEVICE)
            y=y.to(DEVICE)
            pred = model(X).flatten()
            test_loss += loss_fn(pred, y).item()
    print("Test loss (avg):",  test_loss)

    test_loss /= num_batches

    return test_loss

### Optimization Process

In [16]:
for epoch in range(epochs):
    print(f"Epoch {epoch+1}\n-------------------------------")
    train_loss = housing_train_loop(train_housing_data_loader, housing_model, housing_loss_fn, housing_optimizer)
    test_loss = housing_test_loop(test_housing_data_loader, housing_model, housing_loss_fn)

    print("Housing_Loss/train", train_loss)
    print("Housing_Loss/test", test_loss)

print("Done!")

Epoch 1
-------------------------------
Train loss: 917.077271  [   50/18576]
Train loss: 500696.625000  [ 5050/18576]
Train loss: 335659.343750  [10050/18576]
Train loss: 224975.234375  [15050/18576]
Test loss (avg): 7075250.28125
Housing_Loss/train tensor(169314.7812, device='cuda:0', grad_fn=<MseLossBackward0>)
Housing_Loss/test 168458.3400297619
Epoch 2
-------------------------------
Train loss: 168300.109375  [   50/18576]
Train loss: 112986.000000  [ 5050/18576]
Train loss: 75582.546875  [10050/18576]
Train loss: 50614.417969  [15050/18576]
Test loss (avg): 1595218.25
Housing_Loss/train tensor(38218.4453, device='cuda:0', grad_fn=<MseLossBackward0>)
Housing_Loss/test 37981.38690476191
Epoch 3
-------------------------------
Train loss: 37907.265625  [   50/18576]
Train loss: 25542.591797  [ 5050/18576]
Train loss: 17072.849609  [10050/18576]
Train loss: 11431.751953  [15050/18576]
Test loss (avg): 359791.859375
Housing_Loss/train tensor(8601.3652, device='cuda:0', grad_fn=<MseLo